# Data Quality and Duplicate Audit

## Scientific objective
Quantify valid molecules, duplicates, label conflicts, prevalence, missing labels, scaffolds, and endpoint overlap before applying a declared duplicate policy.

## Inputs
- `data/interim/standardized_long.csv`

## Expected outputs
- `data/metadata/data_quality_audit.csv`
- `data/metadata/duplicate_resolution_policy.json`
- `data/processed/endpoint_records.csv`
- conflict-policy sensitivity table

## Dependencies
pandas, NumPy

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
The primary policy removes molecule–endpoint groups with contradictory observed labels. Missing labels remain missing.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Majority vote may be scientifically inappropriate when assay protocols differ; quality filtering cannot be used without quality metadata.

## Next notebook
[06_exploratory_data_analysis.ipynb](./06_exploratory_data_analysis.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'smoke', 'seed': 20260723}


In [2]:
from toxicity_screening.pipeline import audit_and_resolve
from toxicity_screening.deduplication import resolve_duplicates
processed = audit_and_resolve(ROOT, policy="remove_conflicts")
audit = pd.read_csv(ROOT / "data/metadata/data_quality_audit.csv")
display(audit)

,endpoint,raw_records,unique_keys,exact_duplicate_rows,duplicate_molecules,conflicting_molecules,valid_records,missing_labels,positive_count,negative_count,positive_prevalence,unique_scaffolds,policy_selected
0,herg_blockade,13445,12966,0,462,17,13445,0,6718,6727,0.499665,5922,remove_conflicts
1,ames_mutagenicity,7278,7248,0,30,2,7278,0,3974,3304,0.546029,1577,remove_conflicts
2,SR-p53,7830,7612,0,161,5,7830,1057,423,6350,0.062454,2271,remove_conflicts
3,SR-ATAD5,7830,7612,0,161,2,7830,759,264,6807,0.037336,2271,remove_conflicts
4,SR-ARE,7830,7612,0,161,16,7830,1999,942,4889,0.161550,2271,remove_conflicts
5,SR-MMP,7830,7612,0,161,8,7830,2020,918,4892,0.158003,2271,remove_conflicts


In [3]:
# Sensitivity analysis reports consequences without changing the primary artifact.

source = pd.read_parquet(
    ROOT / "data" / "interim" / "standardized_long.parquet"
)

source = source.loc[
    source["standardization_status"] == "success"
].copy()

rows = []

for endpoint, group in source.groupby("endpoint", sort=False):
    for policy in ["remove_conflicts", "majority_vote", "uncertain"]:
        resolved = resolve_duplicates(
            group,
            key="molecule_id",
            label="label",
            policy=policy,
        )

        rows.append(
            {
                "endpoint": endpoint,
                "policy": policy,
                "records": len(resolved),
                "observed_labels": int(resolved["label"].notna().sum()),
            }
        )

sensitivity = pd.DataFrame(rows)

sensitivity.to_csv(
    ROOT / "data" / "metadata" / "duplicate_policy_sensitivity.csv",
    index=False,
)

assert processed.groupby(["endpoint", "molecule_id"]).size().max() == 1

display(sensitivity)

,endpoint,policy,records,observed_labels
0,herg_blockade,remove_conflicts,12949,12949
1,herg_blockade,majority_vote,12966,12950
2,herg_blockade,uncertain,12966,12949
3,ames_mutagenicity,remove_conflicts,7246,7246
4,ames_mutagenicity,majority_vote,7248,7246
5,ames_mutagenicity,uncertain,7248,7246
6,SR-p53,remove_conflicts,7607,6583
7,SR-p53,majority_vote,7612,6584
8,SR-p53,uncertain,7612,6583
9,SR-ATAD5,remove_conflicts,7610,6869


### Completion gate
Confirm that the declared artifacts exist before continuing to `06_exploratory_data_analysis.ipynb`.